# Colab Training: Anomaly Detection

**How to use:**
1. Go to **Runtime > Change runtime type > TPU** (free tier works)
2. Paste your Kaggle API key in `KAGGLE_KEY` below
3. **Run All** (Ctrl+F9) — that's it

The 2 shell commands below handle everything: clone, install, config, dataset pull, training, eval, and export.

In [ ]:
import os, json
from pathlib import Path

# ──────────────── EDIT THESE ────────────────
GITHUB_REPO_URL  = "https://github.com/AakashBhat1/collab_training_anomaly_detection.git"
GITHUB_BRANCH    = "main"
KAGGLE_DATASET   = "webadvisor/real-time-anomaly-detection-in-cctv-surveillance"
KAGGLE_USERNAME  = "aakash1233445"
KAGGLE_KEY       = ""               # <-- paste your Kaggle API key here
REPO_DIR         = "/content/intruder_detection_system"
# ─────────────────────────────────────────────

# Mount Google Drive (checkpoint backup + resume across sessions)
from google.colab import drive
drive.mount("/content/drive")

# Set up Kaggle credentials — tries Colab Secrets first, falls back to KAGGLE_KEY above
kaggle_key = ""
try:
    from google.colab import userdata
    kaggle_key = userdata.get("kagle_key")
except Exception:
    pass
if not kaggle_key:
    kaggle_key = KAGGLE_KEY

if kaggle_key and KAGGLE_USERNAME:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"]      = kaggle_key
    # Also write kaggle.json so the kaggle CLI picks it up
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "kaggle.json").write_text(
        json.dumps({"username": KAGGLE_USERNAME, "key": kaggle_key})
    )
    (kaggle_dir / "kaggle.json").chmod(0o600)
    print(f"Kaggle credentials ready for: {KAGGLE_USERNAME}")
else:
    print("WARNING: No Kaggle key found.")
    print("Either paste it in KAGGLE_KEY above, or add 'kagle_key' in Colab Secrets.")

In [ ]:
# Step 1: Clone repo + install deps + update config + pull Kaggle dataset
import subprocess
from pathlib import Path

if not Path(f"{REPO_DIR}/.git").exists():
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch",
                     GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

!bash {REPO_DIR}/collab_scripts/scripts/colab_setup.sh \
    {GITHUB_REPO_URL} \
    --branch {GITHUB_BRANCH} \
    --repo-dir {REPO_DIR} \
    --kaggle-dataset {KAGGLE_DATASET} \
    --kaggle-clean

In [ ]:
# Step 2: Train — split dataset, train model, evaluate, export to ONNX/OpenVINO
!bash {REPO_DIR}/collab_scripts/scripts/colab_run_training.sh {REPO_DIR}